I import datetime for timestamps and Pandas for data processing. EXPECTED_GOLD_COLUMNS defines the column structure that I expect in the final Gold table.


log_pipeline_result() creates a log file whenever a pipeline stage succeeds or fails. It stores the timestamp, pipeline name, stage, status, and message in ADLS.


This function reads the Gold Delta table and converts it into a Pandas DataFrame. It also handles the earlier _c0, _c1 generic-column issue by checking whether the first row contains the actual column names and correcting the DataFrame if necessary.

In [0]:
from datetime import datetime
import pandas as pd

EXPECTED_GOLD_COLUMNS = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "segment",
    "order_id",
    "order_date",
    "product_id_x",
    "quantity",
    "order_status",
    "product_id_y",
    "unit_price",
    "discount",
    "sales_amount",
    "payment_method",
    "gross_amount"
]


def log_pipeline_result(stage, status, message):

    timestamp = datetime.now()
    timestamp_text = timestamp.strftime("%Y%m%d_%H%M%S")

    log_path = (
        "/Volumes/customersprocess/default/"
        f"customer_sales_logs/"
        f"{stage}_{status}_{timestamp_text}.csv"
    )

    log_data = pd.DataFrame([
        {
            "timestamp": timestamp,
            "pipeline": "customer_sales_etl",
            "stage": stage,
            "status": status,
            "message": message
        }
    ])

    log_data.to_csv(log_path, index=False)

    print(f"Log written: {log_path}")


def load_gold_table():

    gold_df = spark.table(
        "customersprocess.default.customer_sales_gold"
    ).toPandas()

    generic_column_names = [
        f"_c{i}"
        for i in range(len(gold_df.columns))
    ]

    if list(gold_df.columns) == generic_column_names:

        if gold_df.empty:
            return pd.DataFrame(columns=EXPECTED_GOLD_COLUMNS)

        first_row = gold_df.iloc[0].tolist()

        if first_row == EXPECTED_GOLD_COLUMNS:
            normalized_df = gold_df.iloc[1:].copy()
            normalized_df.columns = EXPECTED_GOLD_COLUMNS
            return normalized_df.reset_index(drop=True)

    return gold_df

In [0]:
gold_df = load_gold_table()

print("Gold table loaded successfully")
print("Gold business row count:", len(gold_df))
print("Gold columns:", gold_df.columns.tolist())

In [0]:
display(gold_df.head(5))

I use a try block to perform the final data quality checks on the Gold table. First, I check whether the Gold table is empty and whether all required columns are present.Next, I convert the required numeric columns and order_date into the correct data types. Then I check for null values and duplicate order_id values. If any validation fails, an exception is raised. If all checks pass, I create a SUCCESS log with the number of rows and print FINAL DQ PASSED. If something fails, the except block creates a FAILED log, prints the error, and raise sends the failure back to Databricks so the Job is marked as faile

In [0]:
try:

    if gold_df.empty:
        raise Exception("Gold table is empty")

    required_columns = [
        "customer_id",
        "customer_name",
        "order_id",
        "product_id_x",
        "quantity",
        "unit_price",
        "sales_amount",
        "gross_amount"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in gold_df.columns
    ]

    if missing_columns:
        raise Exception(
            f"Missing required columns: {missing_columns}"
        )

    numeric_columns = [
        "quantity",
        "unit_price",
        "discount",
        "sales_amount",
        "gross_amount"
    ]

    for column in numeric_columns:
        if column in gold_df.columns:
            gold_df[column] = pd.to_numeric(
                gold_df[column],
                errors="coerce"
            )

    if "order_date" in gold_df.columns:
        gold_df["order_date"] = pd.to_datetime(
            gold_df["order_date"],
            errors="coerce"
        )

    null_counts = gold_df[required_columns].isnull().sum()
    failing_columns = null_counts[null_counts > 0]

    if not failing_columns.empty:
        raise Exception(
            "Columns with null values: "
            + ", ".join(
                [
                    f"{column}={count}"
                    for column, count in failing_columns.items()
                ]
            )
        )

    duplicate_order_count = gold_df["order_id"].duplicated().sum()

    if duplicate_order_count > 0:
        raise Exception(
            "Gold table contains duplicate order_id values"
        )

    log_pipeline_result(
        "final_dq",
        "SUCCESS",
        f"Final DQ validation passed for {len(gold_df)} rows"
    )

    print("FINAL DQ PASSED")

except Exception as e:

    log_pipeline_result(
        "final_dq",
        "FAILED",
        str(e)
    )

    print(f"FINAL DQ FAILED: {e}")

    raise